Loading data into dataframe

In [1]:
!pip install pandas
!pip install nltk
!pip install matplotlib
!pip install scikit-learn

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
import re
import string
import nltk
import matplotlib.pyplot as plt
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from itertools import compress
from nltk.stem import WordNetLemmatizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
# Fetch dataset from Firebase instead of local CSV\nimport sys\nimport os\nif os.path.abspath('.') not in sys.path: sys.path.append(os.path.abspath('.'))\nfrom firebase_manager import fetch_dataframe_from_firebase\ntext = fetch_dataframe_from_firebase('reduced_reviews')

Sentiment Analysis

In [4]:
!pip install vaderSentiment


Recommender

In [5]:
# Install SpaCy
!pip install spacy

# Download the medium English model
!python -m spacy download en_core_web_md


     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     -- ------------------------------------- 2.1/33.5 MB 6.9 MB/s eta 0:00:05
     ---- ----------------------------------- 3.4/33.5 MB 8.8 MB/s eta 0:00:04
     ---- ----------------------------------- 3.4/33.5 MB 8.8 MB/s eta 0:00:04
     ------ --------------------------------- 5.8/33.5 MB 6.2 MB/s eta 0:00:05
     ------- -------------------------------- 6.6/33.5 MB 5.8 MB/s eta 0:00:05
     -------- ------------------------------- 7.3/33.5 MB 5.6 MB/s eta 0:00:05
     ---------- ----------------------------- 8.9/33.5 MB 5.8 MB/s eta 0:00:05
     -------------- ------------------------- 12.1/33.5 MB 7.3 MB/s eta 0:00:03
     --------------- ------------------------ 12.8/33.5 MB 6.5 MB/s eta 0:00:04
     ------------------ --------------------- 15.2/33.5 MB 7.1 MB/s eta 0:00:03
     ------------------- -------------------- 16.5/33.5 MB 7.1 

In [6]:
# Fetch dataset from Firebase instead of local CSV\nimport sys\nimport os\nif os.path.abspath('.') not in sys.path: sys.path.append(os.path.abspath('.'))\nfrom firebase_manager import fetch_dataframe_from_firebase\ntext = fetch_dataframe_from_firebase('reduced_reviews')

In [7]:
import spacy
nlp = spacy.load('en_core_web_md')

In [8]:
text['Simplified City'] = text['City'].apply(lambda x: re.sub(r"[^A-Za-z]+", ' ',x.lower()).split())

In [9]:
# Get city input
city=list()
City = input('Enter your preferred city. If no preference, just press Enter \n').lower()
city.append(City)

Enter your preferred city. If no preference, just press Enter 
 jaipur


In [10]:
# Make dynamic after finishing code
# attr_list = ['park','fun','children']
attr_list = []
print('Enter what attribute you would want in your destination. Once you\'re done, just press Enter\n')
while 1:
    word = input().lower()
    if word!='':
        attr_list.append(word)
    else:
        break

attr_list

Enter what attribute you would want in your destination. Once you're done, just press Enter



 heritage
 


['heritage']

In [11]:
text['Review'] = text['Review'].fillna('')

In [12]:
text['Spacy Similarity'] = text['Review'].apply(lambda x: nlp(' '.join(attr_list)).similarity(nlp(x)))

In [ ]:
text.to_csv("C:/Users/cheta/Desktop/IDP-main/Processed_Review_db.csv")

In [13]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
text['Sentiment'] = text['Review'].apply(lambda x: analyzer.polarity_scores(str(x))['compound'])


In [14]:
# Compound score
text['Spacy Similarity x Sentiment'] = text['Spacy Similarity'] * text['Sentiment']

In [15]:
agg_text = text.copy()
#agg_text = agg_text.drop(['Popular Mentions','Review','Words'], axis=1)
agg_text = agg_text.drop(['Raw_Review'], axis=1)
#aggregation_functions = {'Country': 'first','City': 'first','Description': 'first','Price': 'first', 'Rating':'mean','review_clean': 'sum', 'Sentiment score': 'mean','Spacy Similarity': 'mean','Spacy Similarity x Sentiment':'mean','Simplified Country':'first'}
aggregation_functions = {'City': 'first', 'Rating':'mean','Review': 'sum', 'Sentiment': 'mean','Spacy Similarity': 'mean','Spacy Similarity x Sentiment':'mean','Simplified City':'first'}
agg_text = agg_text.groupby(text['Place']).aggregate(aggregation_functions)
# agg_text = agg_text.sort_values(by='Spacy Similarity', ascending=False)
agg_text = agg_text.sort_values(by='Spacy Similarity x Sentiment', ascending=False)
agg_text['match']=round((agg_text['Spacy Similarity'] * 100), 2)
agg_text[:5]

,City,Rating,Review,Sentiment,Spacy Similarity,Spacy Similarity x Sentiment,Simplified City,match
Place,,,,,,,,
Vaikom Mahadeva Temple,Kottayam,5.0,visit temples like feel proud born india rich ...,0.9393,0.668640,0.628054,[kottayam],66.86
Ayyanar Horse Temple,Kanadukathan,5.0,visited tamil new year festival temple stands ...,0.9169,0.663414,0.608284,[kanadukathan],66.34
Ayya Vaikundar Nizhal Thangal,Kanyakumari,5.0,temple certain people come worship special imp...,0.8750,0.660917,0.578302,[kanyakumari],66.09
Fisherfolk Temple,Thalassery,5.0,promising believed locals thalassery people be...,0.8710,0.659859,0.574737,[thalassery],65.99
Bara Pathar Temple,Dalhousie,5.0,ancient temple enjoy view peaceful relaxing mi...,0.9246,0.610649,0.564606,[dalhousie],61.06


In [16]:
recommendations = pd.DataFrame()
recommendations.shape

(0, 0)

In [17]:
if city[0] != '':
    filt_text = agg_text[agg_text['Simplified City'].apply(lambda x: city[0] in x)]
else:
    filt_text = agg_text

In [18]:
print('Based on the attributes you\'ve entered, our top recommendation is', ''.join([i for i in filt_text.index[0] if (i.isalpha() or i.isspace())]))
print('Here are some details about',''.join([i for i in filt_text.index[0] if (i.isalpha() or i.isspace())]))
#print('Country: ',filt_text.iloc[0]['Country'])
print('City: ',filt_text.iloc[0]['City'])
top_city=filt_text.iloc[0]['City']
city.append(top_city)
#print('Description: ',filt_text.iloc[0]['Description'])
#print('Price: ',filt_text.iloc[0]['Price'])
print('Average Rating: ',filt_text.iloc[0]['Rating'])
print('% match: ',round(filt_text.iloc[0]['Spacy Similarity'] * 100, 2), '%')
#if review_desc.loc[filt_text.index[0], 'Similarity'] != 0:
 #   print('Here\'s how similar the reviews are to the official description ', round(review_desc.loc[filt_text.index[0], 'Similarity'] * 100, 2), '%')

Based on the attributes you've entered, our top recommendation is Ele Experience Farm
Here are some details about Ele Experience Farm
City:  Jaipur
Average Rating:  5.0
% match:  49.53 %


In [19]:
filt_text_city = agg_text[agg_text['City'].apply(lambda x: city[1] in x)]
filt_text_city

,City,Rating,Review,Sentiment,Spacy Similarity,Spacy Similarity x Sentiment,Simplified City,match
Place,,,,,,,,
Ele Experience Farm,Jaipur,5.000000,hello thanks words elephant farm pleasure host...,0.936000,0.495310,0.463610,[jaipur],49.53
Creative Fashion India,Jaipur,5.000000,say thought dealing reputable people anuj brot...,0.836000,0.505231,0.422373,[jaipur],50.52
Sawai Mansingh Stadium,Jaipur,5.000000,sawai man singh stadium good place players pla...,0.933700,0.448219,0.418502,[jaipur],44.82
Shila Devi Temple,Jaipur,5.000000,visiting amer fort make sure visit temple couy...,0.845150,0.486078,0.407228,[jaipur],48.61
Central Park,Jaipur,5.000000,mornings feels like heaven earth fresh air bir...,0.777150,0.521659,0.404719,[jaipur],52.17
Elephantwala,Jaipur,5.000000,time elephant ride super experience went cousi...,0.948500,0.422074,0.400338,[jaipur],42.21
Galtaji Temple,Jaipur,4.500000,woods outside jaipur beautiful religious compl...,0.781450,0.502977,0.398090,[jaipur],50.30
Jawahar Kala Kendra,Jaipur,5.000000,went night colleague govind family thousand ni...,0.848100,0.468799,0.397589,[jaipur],46.88
Elephant Joy,Jaipur,5.000000,came group elephant farm great experience feed...,0.875950,0.449020,0.395570,[jaipur],44.90


In [20]:
def similar_places(df):

  if(len(df)==1):
    print('Sorry... There are no other places matching the description in the vicinity\n')
  elif(3>len(df)):
    print('Here are some similar spots worth checking out in your vicinity\n')
    return filt_text_city[1:len(df)]
  else:
    print('Here are some similar spots worth checking out in your vicinity\n')
    return filt_text_city[1:6]

In [21]:
similar_places(filt_text_city)

Here are some similar spots worth checking out in your vicinity



,City,Rating,Review,Sentiment,Spacy Similarity,Spacy Similarity x Sentiment,Simplified City,match
Place,,,,,,,,
Creative Fashion India,Jaipur,5.0,say thought dealing reputable people anuj brot...,0.83600,0.505231,0.422373,[jaipur],50.52
Sawai Mansingh Stadium,Jaipur,5.0,sawai man singh stadium good place players pla...,0.93370,0.448219,0.418502,[jaipur],44.82
Shila Devi Temple,Jaipur,5.0,visiting amer fort make sure visit temple couy...,0.84515,0.486078,0.407228,[jaipur],48.61
Central Park,Jaipur,5.0,mornings feels like heaven earth fresh air bir...,0.77715,0.521659,0.404719,[jaipur],52.17
Elephantwala,Jaipur,5.0,time elephant ride super experience went cousi...,0.94850,0.422074,0.400338,[jaipur],42.21


In [22]:
print('Our next recommendation is', ''.join([i for i in filt_text.index[1] if (i.isalpha() or i.isspace())]))
print('Here are some details about',''.join([i for i in filt_text.index[1] if (i.isalpha() or i.isspace())]))
#print('Country: ',filt_text.iloc[1]['Country'])
print('City: ',filt_text.iloc[1]['City'])
top_city = filt_text.iloc[1]['City']
city.append(top_city)
#print('Description: ',filt_text.iloc[1]['Description'])
#print('Price: ',filt_text.iloc[1]['Price'])
print('Average Rating: ',filt_text.iloc[1]['Rating'])
print('% match: ',round(filt_text.iloc[1]['Spacy Similarity'] * 100, 2), '%')
#if review_desc.loc[filt_text.index[1], 'Similarity'] != 0:
#    print('Here\'s how similar the reviews are to the official description ', round(review_desc.loc[filt_text.index[1], 'Similarity'] * 100,2), '%')

Our next recommendation is Creative Fashion India
Here are some details about Creative Fashion India
City:  Jaipur
Average Rating:  5.0
% match:  50.52 %


In [23]:
filt_text_city = agg_text[agg_text['City'].apply(lambda x: city[2] in x)]
filt_text_city

,City,Rating,Review,Sentiment,Spacy Similarity,Spacy Similarity x Sentiment,Simplified City,match
Place,,,,,,,,
Ele Experience Farm,Jaipur,5.000000,hello thanks words elephant farm pleasure host...,0.936000,0.495310,0.463610,[jaipur],49.53
Creative Fashion India,Jaipur,5.000000,say thought dealing reputable people anuj brot...,0.836000,0.505231,0.422373,[jaipur],50.52
Sawai Mansingh Stadium,Jaipur,5.000000,sawai man singh stadium good place players pla...,0.933700,0.448219,0.418502,[jaipur],44.82
Shila Devi Temple,Jaipur,5.000000,visiting amer fort make sure visit temple couy...,0.845150,0.486078,0.407228,[jaipur],48.61
Central Park,Jaipur,5.000000,mornings feels like heaven earth fresh air bir...,0.777150,0.521659,0.404719,[jaipur],52.17
Elephantwala,Jaipur,5.000000,time elephant ride super experience went cousi...,0.948500,0.422074,0.400338,[jaipur],42.21
Galtaji Temple,Jaipur,4.500000,woods outside jaipur beautiful religious compl...,0.781450,0.502977,0.398090,[jaipur],50.30
Jawahar Kala Kendra,Jaipur,5.000000,went night colleague govind family thousand ni...,0.848100,0.468799,0.397589,[jaipur],46.88
Elephant Joy,Jaipur,5.000000,came group elephant farm great experience feed...,0.875950,0.449020,0.395570,[jaipur],44.90


In [24]:
result=similar_places(filt_text_city)
result

Here are some similar spots worth checking out in your vicinity



,City,Rating,Review,Sentiment,Spacy Similarity,Spacy Similarity x Sentiment,Simplified City,match
Place,,,,,,,,
Creative Fashion India,Jaipur,5.0,say thought dealing reputable people anuj brot...,0.83600,0.505231,0.422373,[jaipur],50.52
Sawai Mansingh Stadium,Jaipur,5.0,sawai man singh stadium good place players pla...,0.93370,0.448219,0.418502,[jaipur],44.82
Shila Devi Temple,Jaipur,5.0,visiting amer fort make sure visit temple couy...,0.84515,0.486078,0.407228,[jaipur],48.61
Central Park,Jaipur,5.0,mornings feels like heaven earth fresh air bir...,0.77715,0.521659,0.404719,[jaipur],52.17
Elephantwala,Jaipur,5.0,time elephant ride super experience went cousi...,0.94850,0.422074,0.400338,[jaipur],42.21


itinerary generator

In [33]:
from google import genai
import pandas as pd

# --- Phase 1: Authentication and Model Initialization ---
# The modern SDK utilizes a Client instantiation rather than global configuration.
# Insert your NEW, secure API key here.
import os
from dotenv import load_dotenv
load_dotenv()
client = genai.Client(api_key=os.environ.get('GOOGLE_API_KEY'))

# --- Phase 2: Data Extraction and Validation ---

# SAFE RESET: Only reset the index if 'Place' isn't already a column.
# This stops the notebook from crashing if you run this cell multiple times!
if 'Place' not in filt_text_city.columns:
    filt_text_city = filt_text_city.reset_index()

# Isolate the primary recommendation (index 0).
result = filt_text_city.iloc[0]

chosen_place = result['Place']
chosen_city = result['City']

print(f"Algorithmically Selected Destination: {chosen_place} situated in {chosen_city}")

# --- Phase 3: Parameter Ingestion ---
# Note: The hardcoded DataFrame containing Mumbai/Delhi/Goa has been permanently excised
# to preserve the integrity of your NLP recommendation engine.
num_days = int(input(f"Please enter the intended duration for the {chosen_place} expedition (in days): "))

# --- Phase 4: Financial Estimation Module ---
def get_price_estimate(client, place, city):
    """Queries the LLM for a generalized financial estimation."""
    prompt = f"Provide a concise financial estimate for a visit to {place} in {city}, accounting for entry fees and primary activities."
    
    # Modern SDK execution protocol
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    return response.text

print(f"\nProcessing financial estimations for {chosen_place}...")
estimated_price = get_price_estimate(client, chosen_place, chosen_city)
print(f"Financial Projection:\n{estimated_price}")

# --- Phase 5: Itinerary Generation Module ---
def generate_itinerary(client, place, city, days, estimated_price):
    """Synthesizes a structured itinerary based on temporal and geographic parameters."""
    prompt = (
        f"Act as a professional logistical coordinator. "
        f"Formulate a structured {days}-day itinerary for a field expedition to {place} located in {city}. "
        f"Integrate the primary activities, strategic time allocations, and accommodate a financial parameter of: {estimated_price}."
    )
    
    # Modern SDK execution protocol
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    return response.text

print("\nSynthesizing the chronological itinerary framework...")
itinerary = generate_itinerary(client, chosen_place, chosen_city, num_days, estimated_price)

print("\n--- Finalized Itinerary ---")
print(itinerary)

Algorithmically Selected Destination: Ele Experience Farm situated in Jaipur


Please enter the intended duration for the Ele Experience Farm expedition (in days):  2



Processing financial estimations for Ele Experience Farm...
Financial Projection:
A visit to Ele Experience Farm, including entry and primary activities like elephant feeding, bathing, and interaction, typically costs **INR 2,500 - 4,000 per person**. This estimate excludes transport and personal expenses.

Synthesizing the chronological itinerary framework...

--- Finalized Itinerary ---
Okay, as your dedicated logistical coordinator, I have formulated a comprehensive and structured 2-day itinerary for your expedition to Ele Experience Farm in Jaipur. This plan focuses on optimal engagement with the primary activity while strategically integrating other key Jaipur highlights to maximize your experience within the specified financial parameters and logistical considerations.

---

**Expedition Title:** Ele Experience & Jaipur Cultural Immersion
**Duration:** 2 Days / 1 Night
**Destination:** Ele Experience Farm, Jaipur & Jaipur City
**Primary Focus:** Ethical Elephant Interaction and 

In [34]:
!pip install streamlit
import streamlit as st
import pandas as pd
from google import genai
# Import your other required libraries (spacy, vaderSentiment, etc.) here

# --- Page Configuration ---
st.set_page_config(page_title="AI Itinerary Generator", layout="centered")
st.title("🌍 Smart Travel Itinerary Generator")
st.write("Enter your preferences below to generate a custom travel plan.")

# --- API Configuration ---
# In Streamlit, it is best practice to store secrets in a .streamlit/secrets.toml file
# For local testing, you can input it via the UI or securely load it.
api_key = st.text_input("Enter your Google Gemini API Key:", type="password")

# --- User Inputs (Frontend UI) ---
# Replacing your standard input() with Streamlit widgets
col1, col2 = st.columns(2)
with col1:
    preferred_city = st.text_input("Preferred Region/State (e.g., Kerala):", "")
with col2:
    attributes = st.text_input("What are you looking for? (e.g., history, nature):", "")

num_days = st.number_input("Duration of the trip (in days):", min_value=1, max_value=30, value=3)

# --- Execution Trigger ---
if st.button("Generate Itinerary"):
    if not api_key:
        st.error("Please provide a valid API key to proceed.")
    else:
        with st.spinner("Analyzing reviews and finding the best destination..."):
            try:
                # 1. Initialize the Client
                client = genai.Client(api_key=api_key)
                
                # ---------------------------------------------------------
                # INSERT YOUR NLP LOGIC HERE
                # (The Spacy and VADER analysis to generate 'filt_text')
                # For this example, I will mock the result of your NLP analysis
                # ---------------------------------------------------------
                
                # Mocking the result of your NLP engine:
                chosen_place = "Amber Fort" 
                chosen_city = "Jaipur"
                
                st.success(f"Algorithmically Selected Destination: **{chosen_place} in {chosen_city}**")
                
                # 2. Financial Estimation
                st.write(f"Calculating financial estimates for {chosen_place}...")
                est_prompt = f"Provide a concise financial estimate for a visit to {chosen_place} in {chosen_city}."
                est_response = client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=est_prompt
                )
                estimated_price = est_response.text
                
                # 3. Itinerary Generation
                st.write("Synthesizing your chronological itinerary...")
                itin_prompt = (
                    f"Act as a professional logistical coordinator. "
                    f"Formulate a structured {num_days}-day itinerary for a field expedition to {chosen_place} located in {chosen_city}. "
                    f"Integrate the primary activities, strategic time allocations, and accommodate a financial parameter of: {estimated_price}."
                )
                itin_response = client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=itin_prompt
                )
                
                # --- Final Output ---
                st.subheader("Your Custom Itinerary")
                st.markdown(itin_response.text) # Render the markdown response nicely
                
            except Exception as e:
                st.error(f"An error occurred during execution: {e}")

2026-04-01 11:24:39.653 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:24:39.658 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:24:40.419 
  command:

    streamlit run C:\Users\cheta\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-01 11:24:40.423 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:24:40.426 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:24:40.428 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:24:40.431 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn